In [33]:
import pandas as pd
df = pd.read_csv("C:\\Users\\USER\\Documents\\Ashwini\\HRS_Rawdata_09-14-2020_Market_Overview_cheapest.csv" ,
                 sep="\t",
                 encoding = 'unicode_escape')
df.head()

In [34]:
df.to_csv('C:\\Users\\USER\\Documents\\Ashwini\\converted\\HRS_Rawdata_09-14-2020_Market_Overview_cheapest_1.csv', 
          sep = ',', 
          header = True,
          encoding='utf-8',
          quoting=1,
          index=False)

In [7]:
import csv
with open('C:\\Users\\USER\\Documents\\HRS_Rawdata_09-14-2020_Market_Overview_2_breakfast_2.csv', 'wb') as file:
    writer = csv.writer(file)
    writer.writerow(df)

In [ ]:
import pandas as pd
df_53150 = pd.read_excel("C:\\Users\\USER\\Documents\\misc\\CLI_CLIENTS_PER_VPOI_LATEST.xlsx" ,
                   sheet_name = 'Sheet1',
                   header=0)
df_53150.columns

In [ ]:
df_53433 = pd.read_excel("C:\\Users\\USER\\Downloads\\AquaDownloads\\53433.xlsx" ,
                   sheet_name = 'Grid Results',
                   header=0)
df_53433.head()

In [ ]:
df_diff = pd.concat([df_53433, df_53150, df_53150]).drop_duplicates(keep=False)
df_diff.count()

In [ ]:
df_53150.to_csv('C:\\Users\\USER\\Documents\\misc\\CSV\\CLI_CLIENTS_PER_VPOI_LATEST.csv', 
          sep = ';', 
          header = True,
          encoding='utf-8',
          index=False)

In [ ]:
df['City_utf'] = df['CITY'].apply(encode(encoding= 'utf-8',errors= 'strict'))

In [ ]:
df_diff.to_excel('C:\\Users\\USER\\Downloads\\AquaDownloads\\set_diff.xlsx', 
                sheet_name='set_diff')

In [ ]:
df_diff.head()

In [ ]:
df_diff.FILTER_REASON.unique()

In [2]:
import pyexasol
import configparser

#Location of the ini file
config = configparser.ConfigParser()
config.read('C:\\Users\\USER\\.spyder-py3\\ExasolDET.ini')

dsn=config['exasolDET']['dsn']
user=config['exasolDET']['user']
pwd=config['exasolDET']['pwd']
schema=config['exasolDET']['schema']
# Exasol connection
connect = pyexasol.connect(dsn=dsn, user=user, password=pwd, schema=schema)

#171243


In [8]:
import pandas as pd
import numpy as np
df_hotel_list = pd.read_csv("C:\\Users\\USER\\Documents\\misc\\CSV\\HOTELLIST.csv" ,
                 sep=",",
                 encoding = 'unicode_escape'
                   )
# df_hotel_list = {'HOTEL_ID':['171243']}

df_hotel_list = pd.DataFrame(df_hotel_list)
#df_hotel_list.head()
df_corsa = pd.DataFrame()
a=[]

In [11]:
df_hotel_list.count()

In [18]:
# PASTE EXASOL QUERY HERE


for  i in range(0, len(df_hotel_list.HOTEL_ID)) :
    sql_query =  """

    SELECT HOTEL_ID, CLIENT_NAME, INVITATION_ON_PROB
    
    from (


select a.hotel_id, a.hotel_name,a.HOTEL_CHAIN_ID, b.PRIO, ch.HOTEL_CHAIN_NAME, b.CATEGORY, b.AVG_RATING , b.CAPACITY, b.ANALYZED_HOTEL_RNP_BY_SOURCING_CLIENTS_2018,
    
    b.OCCUPANCY_RATE_BY_SOURCING_CLIENTS_2018,
    b.CONTRACT_STATUS, b.HOTEL_STATUS, b.CITY_ID, b.CITY_NAME, b.RFP_DESTINATION, b.COUNTRY_ID, b.COUNTRY_NAME, b.COUNTRY_CODE, b.WORLD_REGION, b.ACCEPTED_SOURCING_CLIENTS_2018, b.HOTEL_CURRENCY, 
     b.AVG_ACCEPTED_RATE_2018 as N_AVG_ACCEPTED_RATE_2018, b.MIN_ACCEPTED_RATE_2018 as N_MIN_ACCEPTED_RATE_2018, 
    
     b.AVG_SUBMITTED_RATE_2018 as N_AVG_SUBMITTED_RATE_2018,  b.MIN_SUBMITTED_RATE_2018
     as N_MIN_SUBMITTED_RATE_2018
     ,cur.CURRENCY_EXCHANGE_RATE *  b.BOTTOM_UP_MIN_TARGET_RATE as N_BOTTOM_UP_MIN_TARGET_RATE,cur.CURRENCY_EXCHANGE_RATE *  b.BOTTOM_UP_MAX_TARGET_RATE as N_BOTTOM_UP_MAX_TARGET_RATE,
     cur.CURRENCY_EXCHANGE_RATE *  b.BOTTOM_UP_AVG_TARGET_RATE as N_BOTTOM_UP_AVG_TARGET_RATE,
    b.SMART_BID, b.CCR, b.SYSTEM_BENCHMARK_RATE, b.DISTANCE_TO_VIRTUAL_POI, c.HOTELS_ALL as POI_AVG_CAPACITY ,
    
    
   cou.CPOICOUNT,
    
    
    b.TARGET_RATE_CURRENCY,    b.TARGET_RATE_LRA_INCL_BF as N_TARGET_RATE_LRA_INCL_BF,
    
   inn.VIRTUAL_POI_NAME , inn.CLIENT_NAME , inn.VPOI_RN_SPELLED_OUT_2018, inn.FORECAST_REQUIRED_HOTELS, inn.SPELLED_OUT_RNS_2018_FORECAST_REQUIRED_HOTELS,
inn.OTHER_PRIO_INCREMENTAL_HOTELS_ON_LEAD_LIST_PER_CLIENTVPOI,
inn.CLIENT_ACCEPTED_AVG_CATEGORY, inn.CLIENT_ACCEPTED_AVG_RATING_HRS_GOOGLE, cur.CURRENCY_EXCHANGE_RATE * coalesce(inn.CLIENT_ACCEPTED_AVG_RATE_2018, 0) as N_CLIENT_ACCEPTED_AVG_RATE_2018,

     cur.CURRENCY_EXCHANGE_RATE * coalesce(inn.CLIENT_ACCEPTED_MIN_RATE_2018 , 0) as N_CLIENT_ACCEPTED_MIN_RATE_2018,
case when inn.hotel_id || ' - ' || inn.client_name = cli.ID then 'Yes' else 'No' end as hotel_invited ,

hot.SUBMITTED_RATE_TYPE , coalesce(hot.SUBMITTED_RATE, 0) as SUBMITTED_RATE , hot.RFP_HOTEL_STATUS_2018,
case when hot.RFP_HOTEL_STATUS_2018 is not null then 
      case when local.N_CLIENT_ACCEPTED_MIN_RATE_2018 = 0 and local.N_CLIENT_ACCEPTED_AVG_RATE_2018 = 0 then 0.5 
     else case when local.N_TARGET_RATE_LRA_INCL_BF < local.N_CLIENT_ACCEPTED_MIN_RATE_2018 then 0.9 
  else      case when local.N_TARGET_RATE_LRA_INCL_BF < local.N_CLIENT_ACCEPTED_AVG_RATE_2018 then 0.8
  else   case when coalesce(local.N_CLIENT_ACCEPTED_AVG_RATE_2018/local.N_TARGET_RATE_LRA_INCL_BF, 0) < 0.5  then 0
  else 0.5 *  (local.N_CLIENT_ACCEPTED_AVG_RATE_2018/local.N_TARGET_RATE_LRA_INCL_BF) 
--    
 end end  end   end end   as    numb,
    
 case when hot.RFP_HOTEL_STATUS_2018 is  null then 


      case when  local.N_CLIENT_ACCEPTED_MIN_RATE_2018 = 0 and local.N_CLIENT_ACCEPTED_AVG_RATE_2018 = 0 then 0.4
         else case when    local.N_TARGET_RATE_LRA_INCL_BF < local.N_CLIENT_ACCEPTED_MIN_RATE_2018 then 0.8 
            else case when    local.N_TARGET_RATE_LRA_INCL_BF < local.N_CLIENT_ACCEPTED_AVG_RATE_2018 then 0.7
                 else  case when   coalesce(local.N_CLIENT_ACCEPTED_AVG_RATE_2018/local.N_TARGET_RATE_LRA_INCL_BF, 0) < 0.5  then 0
                else  0.25 *  (local.N_CLIENT_ACCEPTED_AVG_RATE_2018/local.N_TARGET_RATE_LRA_INCL_BF)
end end  end   end end   as    numb2,


 case when hot.RFP_HOTEL_STATUS_2018 is not null then  local.numb else local.numb2 end as INVITED_PROB,

 local.INVITED_PROB * 100 as INVITATION_ON_PROB, 
 
 SPELLED_OUT_RNS_2018_FORECAST_REQUIRED_HOTELS * local.INVITED_PROB as PROB_AMEND,

 case when local.INVITATION_ON_PROB >= 50  then 1 else 0 end as count_PROB_AMEND



    from DWHBIL.V_LKP_HOTEL a 
left join TEMP.CLI_MP_LEAD_LIST b on a.hotel_id = b.HOTEL_ID
left join TEMP.CLI_POI_TABLE c on b.VIRTUAL_POI_NAME = c.VIRTUAL_POI_NAME
left join (

select count(*) as CPOICOUNT, VIRTUAL_POI_NAME from TEMP.CLI_MP_LEAD_LIST k where PRIO <=3 group by VIRTUAL_POI_NAME

) cou on cou.VIRTUAL_POI_NAME = b.VIRTUAL_POI_NAME



join (

 
 
select a.hotel_id ,c.VIRTUAL_POI_NAME , cpo.CLIENT_NAME , cpo.VPOI_RN_SPELLED_OUT_2018, cpo.FORECAST_REQUIRED_HOTELS, cpo.SPELLED_OUT_RNS_2018_FORECAST_REQUIRED_HOTELS,
cpo.OTHER_PRIO_INCREMENTAL_HOTELS_ON_LEAD_LIST_PER_CLIENTVPOI,
cpo.CLIENT_ACCEPTED_AVG_CATEGORY, cpo.CLIENT_ACCEPTED_AVG_RATING_HRS_GOOGLE, cpo.CLIENT_NAME || ' - ' || c.VIRTUAL_POI_NAME as CLI_VPOI, hc.CLIENT_ACCEPTED_AVG_RATE_2018, hc.CLIENT_ACCEPTED_MIN_RATE_2018




  from   DWHBIL.V_LKP_HOTEL a 
 join TEMP.CLI_MP_LEAD_LIST b on a.hotel_id = b.HOTEL_ID
join TEMP.CLI_POI_TABLE c on b.VIRTUAL_POI_NAME = c.VIRTUAL_POI_NAME
join TEMP.CLI_ACCEPTED_CLIENT ac on ac.HOTEL_ID =  a.HOTEL_ID and c.VIRTUAL_POI_NAME = ac.VPOI --and ac.CLIENT_NAME ='CRRC'
   join TEMP.CLI_CLIENTS_PER_VPOI_LATEST cpo on cpo.VPOI_NAME = c.VIRTUAL_POI_NAME  and replace(ac.CLIENT_NAME, CHAR(13), '') <> replace(cpo.VPOI_NAME_CLIENT_NAME, CHAR(13), '')
   join  TEMP.CLI_HOTEL_CLIENT hc on hc.CLIENT_VPOI = cpo.CLIENT_NAME || ' - ' || c.VIRTUAL_POI_NAME --and hc.HOTEL_ID = a.HOTEL_ID
  
   
where cpo.CLIENT_NAME not in (
select -- a.HOTEL_ID,  c.VIRTUAL_POI_NAME,
cpo.CLIENT_NAME
--case when ( cpo.VPOINAME = b.VIRTUAL_POI_NAME and    count(cpo.VPOINAME) = 0 ) then 1 else 0 end  as "Incremental Customer for Input HKEY"
    from DWHBIL.V_LKP_HOTEL a 
 join TEMP.CLI_MP_LEAD_LIST b on a.hotel_id = b.HOTEL_ID
join TEMP.CLI_POI_TABLE c on b.VIRTUAL_POI_NAME = c.VIRTUAL_POI_NAME
join TEMP.CLI_ACCEPTED_CLIENT ac on ac.HOTEL_ID =  a.HOTEL_ID and c.VIRTUAL_POI_NAME = ac.VPOI --and ac.CLIENT_NAME ='CRRC'
   join TEMP.CLI_CLIENTS_PER_VPOI_LATEST cpo on cpo.VPOI_NAME = c.VIRTUAL_POI_NAME  and replace(ac.CLIENT_NAME, CHAR(13), '') = replace(cpo.CLIENT_NAME, CHAR(13), '')

 where 
 
a.hotel_id ="""+ str(df_hotel_list.HOTEL_ID[i])+"""
  group by cpo.CLIENT_NAME) 
 and  a.hotel_id ="""+ str(df_hotel_list.HOTEL_ID[i])+"""
 
  group by a.hotel_id ,c.VIRTUAL_POI_NAME , cpo.CLIENT_NAME , cpo.VPOI_RN_SPELLED_OUT_2018, cpo.FORECAST_REQUIRED_HOTELS, cpo.SPELLED_OUT_RNS_2018_FORECAST_REQUIRED_HOTELS,
cpo.OTHER_PRIO_INCREMENTAL_HOTELS_ON_LEAD_LIST_PER_CLIENTVPOI,
cpo.CLIENT_ACCEPTED_AVG_CATEGORY, cpo.CLIENT_ACCEPTED_AVG_RATING_HRS_GOOGLE, cpo.CLIENT_NAME || '-' || c.VIRTUAL_POI_NAME,
hc.CLIENT_ACCEPTED_AVG_RATE_2018, hc.CLIENT_ACCEPTED_MIN_RATE_2018
--hc.SUBMITTED_RATE_TYPE --, hc.SUBMITTED_RATE, hc.RFP_HOTEL_STATUS_2018

 ) inn on inn.hotel_id = a.hotel_id 
 
 
 
 left  join (

 
 
select hc.SUBMITTED_RATE_TYPE , hc.SUBMITTED_RATE, hc.RFP_HOTEL_STATUS_2018 ,



a.hotel_id ,c.VIRTUAL_POI_NAME , cpo.CLIENT_NAME 
  from   DWHBIL.V_LKP_HOTEL a 
 join TEMP.CLI_MP_LEAD_LIST b on a.hotel_id = b.HOTEL_ID
join TEMP.CLI_POI_TABLE c on b.VIRTUAL_POI_NAME = c.VIRTUAL_POI_NAME
join TEMP.CLI_ACCEPTED_CLIENT ac on ac.HOTEL_ID =  a.HOTEL_ID and c.VIRTUAL_POI_NAME = ac.VPOI --and ac.CLIENT_NAME ='CRRC'
   join TEMP.CLI_CLIENTS_PER_VPOI_LATEST cpo on cpo.VPOI_NAME = c.VIRTUAL_POI_NAME  and replace(ac.CLIENT_NAME, CHAR(13), '') <> replace(cpo.VPOI_NAME_CLIENT_NAME, CHAR(13), '')
   join  TEMP.CLI_HOTEL_CLIENT hc on hc.id = a.HOTEL_ID || ' - ' ||  cpo.CLIENT_NAME 
  
   
where cpo.CLIENT_NAME not in (
select -- a.HOTEL_ID,  c.VIRTUAL_POI_NAME,
cpo.CLIENT_NAME
--case when ( cpo.VPOINAME = b.VIRTUAL_POI_NAME and    count(cpo.VPOINAME) = 0 ) then 1 else 0 end  as "Incremental Customer for Input HKEY"
    from DWHBIL.V_LKP_HOTEL a 
 join TEMP.CLI_MP_LEAD_LIST b on a.hotel_id = b.HOTEL_ID
join TEMP.CLI_POI_TABLE c on b.VIRTUAL_POI_NAME = c.VIRTUAL_POI_NAME
join TEMP.CLI_ACCEPTED_CLIENT ac on ac.HOTEL_ID =  a.HOTEL_ID and c.VIRTUAL_POI_NAME = ac.VPOI --and ac.CLIENT_NAME ='CRRC'
   join TEMP.CLI_CLIENTS_PER_VPOI_LATEST cpo on cpo.VPOI_NAME = c.VIRTUAL_POI_NAME  and replace(ac.CLIENT_NAME, CHAR(13), '') = replace(cpo.CLIENT_NAME, CHAR(13), '')

 where 

  a.hotel_id ="""+ str(df_hotel_list.HOTEL_ID[i])+"""

  group by cpo.CLIENT_NAME) 
 and     a.hotel_id ="""+ str(df_hotel_list.HOTEL_ID[i])+"""

  group by a.hotel_id ,c.VIRTUAL_POI_NAME , cpo.CLIENT_NAME ,
hc.SUBMITTED_RATE_TYPE , hc.SUBMITTED_RATE, hc.RFP_HOTEL_STATUS_2018

 ) hot on hot.hotel_id = a.hotel_id and hot.CLIENT_NAME =  inn.CLIENT_NAME

 
 
 
 
 
join   TEMP.CLI_CURRENCY_CLI cur on cur.CURRENCY_ISO = TARGET_RATE_CURRENCY
left join TEMP.CLI_HOTEL_CLIENT cli on cli.HOTEL_ID = a.HOTEL_ID and inn.CLI_VPOI = cli.CLIENT_VPOI
  join DWHBIL.LKP_HOTEL_CHAIN ch on ch.HOTEL_CHAIN_ID = a.HOTEL_CHAIN_ID
where a.hotel_id ="""+ str(df_hotel_list.HOTEL_ID[i])+"""

group by 
a.hotel_id, a.hotel_name, a.hotel_chain_id, b.PRIO, ch.HOTEL_CHAIN_NAME, b.CATEGORY, b.AVG_RATING , b.CAPACITY, b.ANALYZED_HOTEL_RNP_BY_SOURCING_CLIENTS_2018,
    
    b.OCCUPANCY_RATE_BY_SOURCING_CLIENTS_2018,
    b.CONTRACT_STATUS, b.HOTEL_STATUS, b.CITY_ID, b.CITY_NAME, b.RFP_DESTINATION, b.COUNTRY_ID, b.COUNTRY_NAME, b.COUNTRY_CODE, b.WORLD_REGION, b.ACCEPTED_SOURCING_CLIENTS_2018, b.HOTEL_CURRENCY, 
    local.N_AVG_ACCEPTED_RATE_2018, local.N_MIN_ACCEPTED_RATE_2018, 
    
     local.N_AVG_SUBMITTED_RATE_2018,  local.N_MIN_SUBMITTED_RATE_2018
     ,local.N_BOTTOM_UP_MIN_TARGET_RATE,local.N_BOTTOM_UP_MAX_TARGET_RATE,
     local.N_BOTTOM_UP_AVG_TARGET_RATE,
    b.SMART_BID, b.CCR, b.SYSTEM_BENCHMARK_RATE, b.DISTANCE_TO_VIRTUAL_POI, c.HOTELS_ALL , cou.CPOICOUNT,
    b.TARGET_RATE_CURRENCY, local.N_TARGET_RATE_LRA_INCL_BF,
    
   inn.VIRTUAL_POI_NAME , inn.CLIENT_NAME , inn.VPOI_RN_SPELLED_OUT_2018, inn.FORECAST_REQUIRED_HOTELS, inn.SPELLED_OUT_RNS_2018_FORECAST_REQUIRED_HOTELS,
inn.OTHER_PRIO_INCREMENTAL_HOTELS_ON_LEAD_LIST_PER_CLIENTVPOI,
inn.CLIENT_ACCEPTED_AVG_CATEGORY, inn.CLIENT_ACCEPTED_AVG_RATING_HRS_GOOGLE, local.N_CLIENT_ACCEPTED_AVG_RATE_2018,

     local.N_CLIENT_ACCEPTED_MIN_RATE_2018, 
local.hotel_invited ,

hot.SUBMITTED_RATE_TYPE , local.SUBMITTED_RATE , hot.RFP_HOTEL_STATUS_2018
     ) az 
"""
    QUERY = connect.execute(sql_query)

    # Importing data into a DataFrame
  
    for row in QUERY:
        a.append(row)
    #print(len(a))
    df_corsa = pd.DataFrame(a)
    df_col_names = QUERY.col_names
    df_corsa.columns = df_col_names
    print(i)

In [3]:
import datetime
 
d = datetime.datetime.today()
df_corsa.to_excel('C:\\Users\\USER\\Documents\\misc\\CSV\\ADR_Imputation_CW50.xlsx', 
          sheet_name='ADR', 
          header = True,
          encoding='utf-8',
          index=False)

In [ ]:
import datetime
 
d = datetime.datetime.today()
d

In [19]:
len(df_corsa.HOTEL_ID.unique())

In [4]:
df.to_excel('C:\\Users\\USER\\Documents\\misc\\CSV\\ADR_Imputation_CW50.xlsx', 
          sheet_name='ADR', 
          header = True,
          encoding='utf-8',
          index=False)